## Agent progress


To stream agent progress, use the @[`stream`][CompiledStateGraph.stream] or @[`astream`][CompiledStateGraph.astream] methods with `stream_mode="updates"`. This emits an event after every agent step.


For example, if you have an agent that calls a tool once, you should see the following updates:

* **LLM node**: @[`AIMessage`] with tool call requests
* **Tool node**: @[`ToolMessage`] with execution result
* **LLM node**: Final AI response



Pass a `thread_id` via `config` so the conversation is checkpointed and follow-up turns can resume the same history. `thread_id` is independent of `stream_mode`; you can also pass `context` alongside it for per-run data your tools read from `runtime.context`.


In [1]:
import urllib.error
import urllib.request
import pprint
from langchain.tools import tool

from langchain.chat_models import init_chat_model

import langchain_groq
import os

from dotenv import load_dotenv

load_dotenv()

True

In [7]:
model_groq_lamma70b = init_chat_model("llama-3.3-70b-versatile",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=10000, temperature=0.0)

model_or = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)

model_or_paid_gpt56_luna_pro = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)


In [3]:
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

In [5]:
agent = create_agent(
    model=model_groq_lamma70b,
    tools=[get_weather],
    checkpointer=InMemorySaver()
)

In [6]:
config = {"configurable": {"thread_id": str(uuid7())}}
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    config=config,
    version="v3",
)
for kind, item in stream.interleave("messages", "tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token, end="", flush=True)
    elif kind == "tool_calls":
        print(f"\nTool call: {item.tool_name}({item.input})")
        for delta in item.output_deltas:
            print(delta, end="", flush=True)
        print(f"\nTool result: {item.output}")

final_state = stream.output 

c:\Users\socgen\ML\agentic_ai_and_ops\.venv\Lib\site-packages\langgraph\pregel\main.py:3708: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
c:\Users\socgen\ML\agentic_ai_and_ops\.venv\Lib\site-packages\langgraph\pregel\main.py:3558: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)



Tool call: get_weather({'city': 'San Francisco'})

Tool result: content="It's always sunny in San Francisco!" name='get_weather' id='682d328d-8544-456f-95aa-7d066034ed99' tool_call_id='0e0zg7kj0'


KeyboardInterrupt: 